# TM1py: Processes and Chores

This module is the seventh chunk of the tm1py course. Readers are
assumed to have absorbed the earlier chunks: the mental model from
chunk 1, connecting and listing from chunk 2, the metadata walk from
chunk 3, reading and writing cells from chunks 4 and 5, and Object
construction from chunk 6.

The audience remains TM1 expert. TurboIntegrator is a familiar tool;
the four code sections (prolog, metadata, data, epilog), the variable
list, the data source types, and the parameter mechanism do not need
to be explained as TM1 concepts. The focus is on how tm1py represents
processes, how to deploy them, and how to run them as part of a
larger orchestration written in Python.

The goal is twofold. First, the learner can author or modify a TI
process from Python and persist it idempotently. Second, the learner
can use `execute_with_return` as a job-runner: trigger a process,
wait for completion, read the structured outcome, and decide what to
do next. These two together cover the bulk of practical TM1
automation work, which is why the chunk plan describes this as a
natural pause point. A learner who finishes here can write end-to-end
automation: read from cubes, transform with pandas, write back, and
trigger TI for the heavy server-side steps.

Chores arrive at the end of the chunk as the scheduled equivalent of
processes. They are TM1's mechanism for chaining processes together
on a recurring schedule, and they appear in tm1py as Objects that
follow the same construction and persistence pattern from chunk 6.

The Sales Plan and What If Plan model from earlier chunks continues
as the running example. The new piece this chunk introduces is a
process called `Refresh_What_If_Plan` that copies the Base scenario
from `Sales Plan` into `What If Plan`, parameterized by period.

---

## Topic list

1. The orchestration scenario
2. The Process object: four code sections and surroundings
3. Authoring a process from Python
4. Persisting: create, update, update_or_create
5. Executing with execute_with_return
6. Parameters
7. Reading the error log when a process fails
8. Chores: scheduled bundles of processes
9. Synchronous vs background execution
10. Real-world design principles
11. Common mistakes

---

## 1. The orchestration scenario

A typical end-to-end automation crosses the line between Python work
and server-side TI work several times. Python reads from a cube,
joins against an external system, computes a result, writes it back,
and asks the server to run a TI that fans the result into the
production cubes. The Python code orchestrates; the TI code does the
work that benefits from running in process with the cube.

The split is not arbitrary. Python's strengths, the rich ecosystem,
the convenience of pandas, the access to external systems, are
exactly the wrong tools for tight loops over millions of cells. TI's
strengths, server-side execution, native cube access, transactional
boundaries, are exactly the wrong tools for joins against SQL,
calls to web APIs, or anything outside TM1. The orchestration
pattern uses each tool for the part of the job it is suited to.

tm1py is what holds the two halves together. It can read and write
cells, which is the part of the job Python is good at, and it can
run TI processes, which is the part of the job TI is good at. A
single Python script that uses both is the canonical shape for
non trivial automation, and the rest of this chunk covers how to
write the TI side of that script.

The running example is a simple process: copy the Base scenario from
the Sales Plan cube into the What If Plan cube for a given period.
The process takes one parameter, the period; the data source is a
view; the data tab walks the view and writes the values into the
target cube. Authoring it as a TI process is straightforward; the
question this chunk answers is how to express that authoring,
deployment, and invocation from Python.

## 2. The Process object: four code sections and surroundings

A TI process maps onto a `Process` Object in tm1py. The class is
`Process`, imported from `TM1py.Objects`, and its attributes
correspond directly to the parts of a process visible in PAW or
Architect's process editor.

The four code sections appear as four string attributes:

- `prolog_procedure`: the prolog tab.
- `metadata_procedure`: the metadata tab.
- `data_procedure`: the data tab.
- `epilog_procedure`: the epilog tab.

Each is the literal TI source code that would appear in the
corresponding tab of the process editor. tm1py does not parse, lint,
or rewrite the TI code; it stores the string and sends it to the
server when the process is persisted. The TI compilation happens on
the server when the process is created or updated, and any syntax
errors surface there.

The surrounding configuration also appears as attributes. Parameters
are added with `add_parameter`, variables with `add_variable`, and
the data source is configured through a small group of
`datasource_*` attributes that mirror the data source pane of the
process editor.

In [ ]:
from TM1py.Objects import Process

p = Process(name="Refresh_What_If_Plan")
p.prolog_procedure = "..."
p.metadata_procedure = "..."
p.data_procedure = "..."
p.epilog_procedure = "..."
p.add_parameter(name="pPeriod", prompt="Period to refresh", value="2026Q1")
p.datasource_type = "TM1CubeView"
p.datasource_data_source_name_for_server = "Sales Plan"
p.datasource_view = "Base_Scenario_For_Refresh"

A process Object on its own is just data. It does not exist on the
server until the matching service persists it (topic 4). This is
the same Object/Service split from chunk 1; nothing about processes
breaks it.

## 3. Authoring a process from Python

Authoring a TI process from Python is mostly authoring TI: the four
code sections are TI source. What Python adds is a way to assemble,
parameterize, and version-control that source as part of a
deployment script. The disciplined pattern is to keep the TI source
in a separate `.ti` or `.txt` file and load it into the Process
Object at construction time.

In [ ]:
from pathlib import Path
from TM1py.Objects import Process

def build_refresh_what_if_plan() -> Process:
    src = Path("processes/refresh_what_if_plan")
    p = Process(name="Refresh_What_If_Plan")
    p.prolog_procedure   = (src / "prolog.ti").read_text()
    p.metadata_procedure = (src / "metadata.ti").read_text()
    p.data_procedure     = (src / "data.ti").read_text()
    p.epilog_procedure   = (src / "epilog.ti").read_text()

    p.add_parameter(name="pPeriod", prompt="Period to refresh", value="2026Q1")

    p.datasource_type = "TM1CubeView"
    p.datasource_data_source_name_for_server = "Sales Plan"
    p.datasource_view = "Base_Scenario_For_Refresh"
    return p

The TI source files live under version control alongside the Python
script. A change to the TI logic is a change in those files, with a
clean diff. The Python script is concerned only with assembly and
deployment, not with the content of any one tab.

For very small processes, embedding the TI as a Python triple-quoted
string is acceptable. For anything beyond a handful of lines per
tab, the file-per-tab pattern keeps both the TI and the Python
readable. The trade-off is the same one that applied to cube rules
text in chunk 6: a string literal is convenient until it is large,
at which point a file is the better home.

## 4. Persisting: create, update, update_or_create

The verbs from chunk 6 carry over without change. `tm1.processes`
is the ProcessService, and it exposes `create`, `update`,
`update_or_create`, and `delete` with the same semantics introduced
for dimensions and cubes.

In [ ]:
with TM1Service(**creds) as tm1:
    tm1.processes.update_or_create(build_refresh_what_if_plan())

For deployment scripts, `update_or_create` is the right default.
Running the script the first time creates the process; running it
again updates with the same definition, which is a no-op when the
TI source is unchanged and a useful incremental update when it has
been edited. The script is idempotent, can be re-run safely, and
produces the same final state regardless of whether the process
already existed.

The TI compilation runs server-side on persistence. A syntax error
in the TI source surfaces as an exception on the
`update_or_create` call, with the line number and column from the
TI compiler in the error message. This is one of the few cases
where the Python side of tm1py is just the messenger; the diagnosis
of the failure is in the TI compiler's hands.

A useful side property of persistence-by-update is that the process
can be edited interactively in PAW, then re-read into Python with
`tm1.processes.get(name)`, then committed back with
`update_or_create` after any necessary parameterization changes.
The Object/Service round trip is symmetrical for processes the same
way it is for any other named entity.

## 5. Executing with execute_with_return

The workhorse method for running a process is
`execute_with_return`. It takes the process name and any parameters,
runs the process synchronously on the server, and returns a
three-tuple describing the outcome.

In [ ]:
with TM1Service(**creds) as tm1:
    success, status, error_log = tm1.processes.execute_with_return(
        process_name="Refresh_What_If_Plan",
        pPeriod="2026Q1",
    )

print(success, status, error_log)
# True 'CompletedSuccessfully' ''

The three return values are:

- `success`: a boolean. `True` when the process finished cleanly,
  `False` when something went wrong.
- `status`: a string from a small fixed set:
  `CompletedSuccessfully`, `CompletedWithMessages`, `Aborted`,
  `HasMinorErrors`, `QuitCalled`. The status is the same value that
  PAW shows in the process execution dialog.
- `error_log`: the name of the server-side error log file produced
  by the run, or an empty string if no log file was produced.
  Topic 7 covers reading the file.

The pattern that makes a Python script use this as a job-runner is
to check `success` and act on it.

In [ ]:
with TM1Service(**creds) as tm1:
    success, status, error_log = tm1.processes.execute_with_return(
        process_name="Refresh_What_If_Plan",
        pPeriod="2026Q1",
    )
    if not success:
        raise RuntimeError(
            f"Refresh_What_If_Plan failed: {status}; log: {error_log}"
        )

`execute_with_return` is synchronous: the call blocks until the
process finishes. For long-running processes, the request can take
minutes, and the underlying HTTP connection must stay open the whole
time. tm1py's session timeout, configurable on `TM1Service` via the
`session_context` and timeout parameters, must accommodate the
expected runtime; a process that takes ten minutes against a
service configured with a five-minute timeout fails with a
connection error rather than a clean process status.

The blocking shape is what makes the method a job-runner. The
caller knows, when the call returns, whether the process succeeded
and can react. This is the right shape for orchestration scripts
where the next step depends on the previous step's outcome.

## 6. Parameters

TI process parameters are passed to `execute_with_return` as keyword
arguments, with names matching the TI parameter names exactly,
including the conventional `p` prefix.

In [ ]:
tm1.processes.execute_with_return(
    process_name="Refresh_What_If_Plan",
    pPeriod="2026Q1",
)

For numeric parameters, pass an `int` or `float`. For string
parameters, pass a `str`. tm1py converts each to the form expected
by the REST endpoint and the TI engine. Names are case sensitive
and must match the parameter names declared on the process; a typo
in the name results in the parameter being silently ignored and the
process running with whatever default value the parameter declared.

When parameters are themselves built up programmatically, for
example from argparse, the natural shape is a dict that gets spread
with `**`:

In [ ]:
parameters = {
    "pPeriod":  args.period,
    "pVersion": args.version,
}
tm1.processes.execute_with_return(
    process_name="Refresh_What_If_Plan",
    **parameters,
)

Spreading a dict with `**` keeps the call site readable and lets the
parameter dictionary be assembled, validated, and logged
separately. The pattern composes naturally with command-line tools,
configuration files, and any source where the parameter set is not
known statically.

For the rare process that takes no parameters, the call is just the
process name.

In [ ]:
tm1.processes.execute_with_return(process_name="Archive_Quarter")

## 7. Reading the error log when a process fails

When a process returns with a non empty `error_log` file name, the
file lives in the TM1 server's logs directory and contains the
diagnostic output from the run: which TI line failed, what the
values of variables were, what error was reported by the function
call. The file is the canonical source of truth for "why did this
process fail," and pulling it back into Python is part of any
non trivial orchestration script.

The method is on the ProcessService too:

In [ ]:
with TM1Service(**creds) as tm1:
    success, status, error_log = tm1.processes.execute_with_return(
        process_name="Refresh_What_If_Plan",
        pPeriod="2026Q1",
    )
    if not success:
        log_text = tm1.processes.get_error_log_file_content(error_log)
        print(log_text)
        raise RuntimeError(f"Refresh_What_If_Plan failed: {status}")

The log content is a string of the same lines that would appear in
the file on disk. Including it in the exception message, or sending
it to the orchestration system's logging pipeline, makes the failure
diagnosable without having to log into the TM1 server and read the
file by hand.

For cleanup, the matching `tm1.processes.delete_error_log_file`
removes the file from the server. Whether to delete is a policy
question: deleting on success keeps the logs directory tidy; keeping
all logs means a forensic record of every run. The right answer
depends on the team's logging conventions, not on tm1py.

## 8. Chores: scheduled bundles of processes

A chore in TM1 is a chain of TI processes plus a schedule. tm1py
exposes chores through `tm1.chores`, the ChoreService, with the
same Object/Service shape as everything else in the library.

In [ ]:
from datetime import datetime
from TM1py.Objects import Chore, ChoreFrequency, ChoreTask

nightly = Chore(
    name="Nightly_Refresh",
    start_time=datetime(2026, 1, 1, 2, 0, 0),
    dst_sensitivity=False,
    active=True,
    execution_mode="MultipleCommit",
    frequency=ChoreFrequency(days=1),
    tasks=[
        ChoreTask(step=0, process_name="Load_Sales",          parameters=[]),
        ChoreTask(step=1, process_name="Refresh_GL",          parameters=[]),
        ChoreTask(step=2, process_name="Refresh_What_If_Plan",
                  parameters=[{"Name": "pPeriod", "Value": "2026Q1"}]),
    ],
)

with TM1Service(**creds) as tm1:
    tm1.chores.update_or_create(nightly)
    tm1.chores.activate(chore_name="Nightly_Refresh")

The fields mirror the chore editor in PAW: a start time, a
frequency, an active flag, an execution mode, and a list of tasks.
Each task names a process and the parameters to pass to it. The
construction and persistence shape is the same as for any other
named Object; only the leaf classes differ.

For ad hoc execution outside the schedule, the verb is `execute`.

In [ ]:
tm1.chores.execute(chore_name="Nightly_Refresh")

`tm1.chores.execute` is the asynchronous counterpart to
`execute_with_return`. The chore is launched on the server and runs
to completion in the background; the call returns immediately,
without waiting and without reporting the outcome of any individual
task. For a chore, this is the right shape: a chain of processes
running for tens of minutes is not something a Python script wants
to block on, and the chore's own status is observable on the server
for any monitoring that needs it.

## 9. Synchronous vs background execution

The asymmetry between `tm1.processes.execute_with_return`
(synchronous, returns the outcome) and `tm1.chores.execute`
(asynchronous, returns immediately) is worth naming.

`execute_with_return` is the right call when the next step in the
script depends on the outcome. The Python side blocks, reads the
status, and decides what to do. This fits orchestration scripts
where the read–transform–write sequence has to know whether the
final TI succeeded before declaring the run complete.

`tm1.chores.execute` is the right call for fire-and-forget
scheduled work that monitors itself. The chore runs server-side,
the script does not need to see its progress, and any failure
surfaces in the server's chore log rather than in the Python
caller's exception handling.

For cases that do not fit either shape cleanly, two adjacent
methods exist. `tm1.processes.execute` (without the `_with_return`
suffix) runs the process server-side without waiting and without
reporting status. It is rare in modern code, because the structured
return is almost always wanted. `tm1.processes.execute_async`
launches the process and returns a handle for later polling; this
is the right shape for very long processes where the script needs
to do other work in the meantime and check back. Both are
operational variants of the same pattern; the canonical choice
remains `execute_with_return` for processes where the outcome
matters, and `tm1.chores.execute` for chained server-side jobs that
manage their own monitoring.

## 10. Real-world design principles

**Default to `execute_with_return`.** The synchronous call with the
structured return is the right shape for almost every Python
orchestration script. Reach for the asynchronous variants only when
the workflow specifically calls for fire-and-forget or for parallel
progress on the Python side.

**Always check `success`.** A process can return cleanly from
tm1py, but with `success=False` because TI itself encountered an
error. A script that ignores the return value will treat that as a
clean run. Wrap every `execute_with_return` in a check, raise a
clear exception on failure, and include the error log content in
the exception message for diagnosis.

**Keep TI source in version-controlled files.** The four code
sections, embedded as Python triple-quoted strings, are hard to
read, hard to diff, and hard to edit. Store them as separate `.ti`
files (or `.txt`, or in a `processes/` directory by name) and load
them into the Process Object at construction time.

**Use `update_or_create` for deployment.** As in chunk 6, the
verb that makes scripts idempotent is `update_or_create`. A
deployment script that uses `create` fails on its second run; one
that uses `update_or_create` always leaves the server in the
expected state.

**Prefer the orchestration split: Python plans, TI executes.** The
canonical division of labour for non trivial automation is for
Python to handle the analytical and integration work and to trigger
TI for the heavy server-side parts. The split makes the Python code
shorter, the TI code faster, and the overall pipeline easier to
reason about.

**Pull the error log into your own logging.** A failed run that
silently leaves a log file on the server is a failure that the team
will investigate by SSHing into TM1. A failed run whose log content
is included in the Python exception, or in the orchestration
system's logs, is one the team can diagnose without leaving the
tools they already have open.

**Parameterize calls, not processes.** Inside Python, the parameter
set lives in a dictionary that is built, logged, and validated.
Inside TI, the same parameters are declared and used. The boundary
between them is the `**parameters` spread on the
`execute_with_return` call. Build the dict once, near where the
inputs are known; spread it at the call site.

## 11. Common mistakes

A short collection of errors that come up while learning to deploy
and run TI processes from Python.

**Ignoring the return value.** A process that fails returns
`success=False` with a status string and an error log. A script
that calls `execute_with_return` without inspecting the return
treats every run as success.

In [ ]:
# Wrong: outcome is discarded
tm1.processes.execute_with_return(process_name="Refresh_What_If_Plan", pPeriod="2026Q1")

# Correct: check the result
success, status, error_log = tm1.processes.execute_with_return(
    process_name="Refresh_What_If_Plan", pPeriod="2026Q1",
)
if not success:
    raise RuntimeError(f"failed with status {status}; log: {error_log}")

**Misspelling a parameter name.** Parameter names are case
sensitive and matched exactly against the TI parameter list. A
misspelled name is silently ignored, and the process runs with the
default value (or none) for the parameter the script meant to
override.

In [ ]:
# Wrong: 'pperiod' (lowercase) does not match TI's 'pPeriod'
tm1.processes.execute_with_return(
    process_name="Refresh_What_If_Plan",
    pperiod="2026Q1",
)
# the process runs with whatever default pPeriod has

# Correct
tm1.processes.execute_with_return(
    process_name="Refresh_What_If_Plan",
    pPeriod="2026Q1",
)

**Embedding TI source as triple-quoted strings.** Acceptable for
small experiments. Painful for any process beyond a few lines per
tab.

In [ ]:
# Wrong: hard to read, hard to diff
p.prolog_procedure = """
sCube = 'What If Plan';
sScenario = 'Base';
nValue = ...
... many more lines ...
"""

# Correct: TI in its own file, version-controlled
p.prolog_procedure = Path("processes/refresh_what_if_plan/prolog.ti").read_text()

**Calling `execute` instead of `execute_with_return`.** Some
versions of tm1py expose a non-`_with_return` variant that does not
report the outcome. Using it in an orchestration script means the
script never finds out about failures.

In [ ]:
# Wrong: no return value, failures invisible
tm1.processes.execute(process_name="Refresh_What_If_Plan", pPeriod="2026Q1")

# Correct: structured return
success, status, error_log = tm1.processes.execute_with_return(
    process_name="Refresh_What_If_Plan", pPeriod="2026Q1",
)

**Treating chore execution as synchronous.** `tm1.chores.execute`
returns immediately. Code that proceeds as if the chore has
finished by the time the call returns will read stale data or run
its next step before the chore's writes are committed.

In [ ]:
# Wrong: chore is still running
tm1.chores.execute(chore_name="Nightly_Refresh")
df = tm1.cells.execute_view_dataframe("Sales Plan", "Updated By Chore")
# df reflects state before the chore finished

# Correct: either wait, or trigger the steps individually with
# execute_with_return so the script can block on each one
for step in steps:
    success, status, error_log = tm1.processes.execute_with_return(
        process_name=step["name"], **step["parameters"],
    )
    if not success:
        raise RuntimeError(f"{step['name']} failed: {status}")
df = tm1.cells.execute_view_dataframe("Sales Plan", "Updated By Process")

**Forgetting that the session must outlive the process.** A
synchronous `execute_with_return` blocks for the duration of the
TI run. A long-running process needs the underlying HTTP session
to stay open for the whole duration. A `with TM1Service(...)` block
that is allowed to exit while the process is still running tears
down the session and aborts the request.

In [ ]:
# Wrong: session closes while process runs (only relevant if the
# block is structured so the call is the last step and a sibling
# thread exits the block early)
def kick_off():
    with TM1Service(**creds) as tm1:
        tm1.processes.execute_with_return(process_name="Slow_Process")

# Correct: keep the session open for the duration; configure
# timeouts on the TM1Service to accommodate the expected runtime.
with TM1Service(**creds, timeout=3600) as tm1:
    success, status, error_log = tm1.processes.execute_with_return(
        process_name="Slow_Process",
    )

**Hardcoding chore start times in source code.** Chore schedules
are configuration, not code. Either keep them in a deployment
configuration file and load them at deployment time, or accept that
the script is environment-specific. Embedding `datetime(2026, 1, 1,
2, 0, 0)` directly in source produces a chore whose schedule is
correct in one timezone and one environment and incorrect
everywhere else.

**Catching `TM1pyException` and continuing.** A bare except that
swallows the exception removes the only signal that a process or
chore deployment failed. Either let the exception propagate, or log
it and re-raise, but do not pretend the failure did not happen.

In [ ]:
# Wrong: failure is invisible
try:
    tm1.processes.update_or_create(p)
except Exception:
    pass

# Correct: log and propagate
try:
    tm1.processes.update_or_create(p)
except TM1pyException as exc:
    logger.error("process deployment failed: %s", exc)
    raise